In [1]:
# ==============================================================================
# Task 2: ADVANCED DATA COLLECTION, CLEANING, AND PREPROCESSING PIPELINE

# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ------------------------------------------------------------------------------
# 1. SIMULATING A REALISTIC, "MESSY" LOGISTICS DATASET
# ------------------------------------------------------------------------------
np.random.seed(42)
n_samples = 5000

print(f"Step 1: Simulating raw dataset containing {n_samples} delivery records...")
raw_logistics_data = {
    'order_id': range(20000, 20000 + n_samples),
    'distance_km': np.random.uniform(1.5, 45.0, n_samples),
    'delivery_time_hrs': np.random.uniform(0.4, 5.5, n_samples),
    'fuel_cost_usd': np.random.uniform(4.0, 75.0, n_samples),
    'package_weight_kg': np.random.uniform(0.2, 25.0, n_samples),
    'vehicle_type': np.random.choice(['Van', 'Bike', 'Truck', np.nan], size=n_samples, p=[0.45, 0.30, 0.20, 0.05]),
    'weather_condition': np.random.choice(['Clear', 'Rain', 'Fog', 'Heavy Traffic', None], size=n_samples, p=[0.50, 0.20, 0.10, 0.15, 0.05])
}

df_raw = pd.DataFrame(raw_logistics_data)

# Inject intentional real-world data quality issues
# A. Missing values in numerical columns
df_raw.loc[np.random.choice(df_raw.index, 250), 'distance_km'] = np.nan
df_raw.loc[np.random.choice(df_raw.index, 180), 'delivery_time_hrs'] = np.nan
df_raw.loc[np.random.choice(df_raw.index, 300), 'package_weight_kg'] = np.nan

# B. Outliers and invalid values (physics violations)
df_raw.loc[12, 'delivery_time_hrs'] = 99.9  # Severe outlier (traffic logging error)
df_raw.loc[45, 'fuel_cost_usd'] = -120.5    # Invalid negative cost
df_raw.loc[102, 'distance_km'] = 500.0      # Erroneous long distance for last-mile

print("\n--- Initial Raw Dataset Profile ---")
print(df_raw.info())
print("\nMissing Values Count per Column:")
print(df_raw.isnull().sum())


# ------------------------------------------------------------------------------
# 2. DATA CLEANING & IMPUTATION PIPELINE
# ------------------------------------------------------------------------------
print("\n--- Step 2: Executing Data Cleaning Pipeline ---")
df_clean = df_raw.copy()

# A. Handling Numerical Missing Values (Using Median to protect against skewness)
for col in ['distance_km', 'delivery_time_hrs', 'fuel_cost_usd', 'package_weight_kg']:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"Imputed missing values in '{col}' with median: {median_val:.2f}")

# B. Handling Categorical Missing Values (Filling with 'Unknown' or Mode)
df_clean['vehicle_type'] = df_clean['vehicle_type'].fillna('Unknown_Vehicle')
df_clean['weather_condition'] = df_clean['weather_condition'].fillna('Clear')

# C. Removing Invalid Values (Physical constraints: cost > 0, distance > 0)
initial_row_count = len(df_clean)
df_clean = df_clean[(df_clean['fuel_cost_usd'] > 0) & (df_clean['distance_km'] > 0)]
print(f"Removed {initial_row_count - len(df_clean)} rows due to invalid physical constraints (negative costs/zero distances).")

# D. Outlier Detection and Filtering using Interquartile Range (IQR)
def filter_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    filtered_data = data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]
    print(f"IQR Outlier removal on '{column}': Dropped {len(data) - len(filtered_data)} rows.")
    return filtered_data

for col in ['distance_km', 'delivery_time_hrs', 'fuel_cost_usd']:
    df_clean = filter_iqr(df_clean, col)


# ------------------------------------------------------------------------------
# 3. FEATURE ENGINEERING
# ------------------------------------------------------------------------------
print("\n--- Step 3: Feature Engineering ---")
# Calculate Average Speed (km/h) and Cost Efficiency (USD per km)
df_clean['avg_speed_kmh'] = df_clean['distance_km'] / df_clean['delivery_time_hrs']
df_clean['cost_per_km'] = df_clean['fuel_cost_usd'] / df_clean['distance_km']
print("Successfully engineered new operational features: 'avg_speed_kmh' and 'cost_per_km'.")


# ------------------------------------------------------------------------------
# 4. NORMALIZATION & FEATURE SCALING
# ------------------------------------------------------------------------------
print("\n--- Step 4: Normalization and Feature Scaling ---")
scaler = MinMaxScaler()
scaling_cols = ['distance_km', 'delivery_time_hrs', 'fuel_cost_usd', 'package_weight_kg', 'avg_speed_kmh', 'cost_per_km']

df_processed = df_clean.copy()
df_processed[scaling_cols] = scaler.fit_transform(df_processed[scaling_cols])

print("\n--- Final Processed Dataset Preview (Normalized 0 to 1) ---")
display(df_processed.head(5))
print(f"Final clean dataset shape: {df_processed.shape}")

Step 1: Simulating raw dataset containing 5000 delivery records...

--- Initial Raw Dataset Profile ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           5000 non-null   int64  
 1   distance_km        4755 non-null   float64
 2   delivery_time_hrs  4822 non-null   float64
 3   fuel_cost_usd      5000 non-null   float64
 4   package_weight_kg  4713 non-null   float64
 5   vehicle_type       5000 non-null   object 
 6   weather_condition  4781 non-null   object 
dtypes: float64(4), int64(1), object(2)
memory usage: 273.6+ KB
None

Missing Values Count per Column:
order_id               0
distance_km          245
delivery_time_hrs    178
fuel_cost_usd          0
package_weight_kg    287
vehicle_type           0
weather_condition    219
dtype: int64

--- Step 2: Executing Data Cleaning Pipeline ---
Imputed missing v

,order_id,distance_km,delivery_time_hrs,fuel_cost_usd,package_weight_kg,vehicle_type,weather_condition,avg_speed_kmh,cost_per_km
0,20000,0.374699,0.393798,0.373615,0.499581,Bike,Clear,0.064534,0.034948
1,20001,0.951134,0.485218,0.332872,0.746739,Van,Clear,0.132852,0.011861
2,20002,0.732315,0.854963,0.176058,0.562599,Van,Rain,0.061055,0.008632
3,20003,0.598919,0.485218,0.607323,0.083077,Bike,Clear,0.084459,0.034838
4,20004,0.156078,0.870073,0.476634,0.185389,Bike,Rain,0.012989,0.096398


Final clean dataset shape: (4997, 9)
